In [ ]:
import os
import sys
import subprocess
import nibabel as nib
import json
import numpy as np
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sub_name = "sub-03"
data_dir = f"{root_dir}/data/{sub_name}/ses-01"
master_template = f"{sub_name}_dir-AP-seepi.nii.gz"

In [ ]:
is_jupyter = any('jupyter' in arg.lower() for arg in sys.argv) or 'ipykernel' in sys.modules

if not is_jupyter:
    data_dir = sys.argv[1] 
    master_template = sys.argv[2] 
    sub_name = sys.argv[3]

# 设置文件路径
ap_file = os.path.join(data_dir, f'{sub_name}_dir-AP-seepi.nii.gz')
pa_file = os.path.join(data_dir, f'{sub_name}_dir-PA-seepi.nii.gz')

# 切换工作目录
os.chdir(data_dir)

# --- 1. 预处理：提取第一帧并计算平均值 ---
subprocess.run(['3dTstat', '-prefix', 'AP_mean.nii.gz', ap_file], check=True)
subprocess.run(['3dTstat', '-prefix', 'PA_mean.nii.gz', pa_file], check=True)

# --- 2. 计算矫正场 (3dQwarp) ---
cmd_qwarp = [
    '3dQwarp', '-plusminus', '-pmbase',
    '-base', 'PA_mean.nii.gz',
    '-source', 'AP_mean.nii.gz',
    '-prefix', 'distort_corr.nii.gz',
    '-noXdis', '-noZdis', '-overwrite'
]
subprocess.run(cmd_qwarp, check=True)

# 1. 设置路径
warp_path = 'distort_corr_PLUS_WARP.nii.gz'
json_path = f'{sub_name}_dir-AP-seepi.json'

# 2. 读取 JSON 参数
with open(json_path, 'r') as f:
    meta = json.load(f)
trt = meta['TotalReadoutTime'] 

# 3. 读取数据
img = nib.load(warp_path)
data = img.get_fdata()
dx, dy, dz = img.header.get_zooms()[:3]

y_displacement_mm = data[..., 1]  # 提取 Y 方向 (索引 1)

# 4. 转换计算 (mm -> pixels -> Hz)
# 公式: (位移mm / 体素大小mm) / TRT
b0_map_hz = (y_displacement_mm / dy) / trt

# 5. 保存
new_img = nib.Nifti1Image(b0_map_hz.astype(np.float32), img.affine)
new_img.header.extensions.clear() 
nib.save(new_img, 'B0_map_Hz_Whole_FOV.nii.gz')

# 将B0 map 重采样到 master 模板的网格和空间
cmd_resample_warp = [
    '3dresample',
    '-master', master_template,
    '-inset', 'B0_map_Hz_Whole_FOV.nii.gz',
    '-prefix', 'B0_map_Hz_to_master.nii.gz',
    '-rmode', 'Cu' 
]
subprocess.run(cmd_resample_warp, check=True)

# --- 3. 生成 Brain Mask ---
subprocess.run(['3dAutomask', '-prefix', 'brain_mask.nii.gz', 'distort_corr.nii.gz'], check=True)

# --- 4. 应用mask到B0 map ---
cmd_calc = [
    '3dcalc',
    '-a', 'B0_map_Hz_to_master.nii.gz',
    '-b', 'brain_mask.nii.gz',
    '-expr', 'a*b',
    '-datum', 'float',
    '-prefix', 'B0_map_Hz_to_master_masked.nii.gz',
    '-overwrite'
]
subprocess.run(cmd_calc, check=True)

img2 = nib.load('B0_map_Hz_to_master_masked.nii.gz')
data2 = img2.get_fdata()

print(f"Hz 范围: {np.min(data2)} 到 {np.max(data2)}")
os.chdir(root_dir)

++ 3dTstat: AFNI version=AFNI_26.0.07 (Jan 24 2026) [64-bit]
++ Authored by: KR Hammett & RW Cox
** AFNI converts NIFTI_datatype=512 (UINT16) in file /mnt/hgfs/MRI/ECC_Project/EPI_shim/NIFTI_sepi/sub-03/ses-01/sub-03_dir-AP-seepi.nii.gz to FLOAT32
     Warnings of this type will be muted for this session.
     Set AFNI_NIFTI_TYPE_WARN to YES to see them all, NO to see none.
++ Output dataset ./AP_mean.nii.gz
++ 3dTstat: AFNI version=AFNI_26.0.07 (Jan 24 2026) [64-bit]
++ Authored by: KR Hammett & RW Cox
** AFNI converts NIFTI_datatype=512 (UINT16) in file /mnt/hgfs/MRI/ECC_Project/EPI_shim/NIFTI_sepi/sub-03/ses-01/sub-03_dir-PA-seepi.nii.gz to FLOAT32
     Warnings of this type will be muted for this session.
     Set AFNI_NIFTI_TYPE_WARN to YES to see them all, NO to see none.
++ Output dataset ./PA_mean.nii.gz
++ OpenMP thread count = 12
++ 3dQwarp: AFNI version=AFNI_26.0.07 (Jan 24 2026) [64-bit]
++ Authored by: Zhark the (Hermite) Cubically Warped
++ Dataset final zero-pad: xbot=11

Hz 范围: -96.6348648071289 到 83.8763198852539


++ Output dataset ./B0_map_Hz_to_master_masked.nii.gz
